In [1]:
import pandas as pd
from app.services.extrai_dados_sped import extrai_dados_sped
from app.services.extrai_dados_planilha_sap import extrai_dados_planilha_sap


sped = extrai_dados_sped(r"C:\Users\luis.moura\Documents\GitHub\projeto-confronto-ajuste-sped\sped_test\PISCOFINS_20260301_20260331_04784935000167_Original_20260415164738_3B13D02B5763C278137AB3286018AD0488132796.txt")
sap = extrai_dados_planilha_sap(r"C:\Users\luis.moura\Documents\GitHub\projeto-confronto-ajuste-sped\diarios_test\Diário 03.2026 - Central.xlsx")

df_f120 = sped['F120']
#print(df_f120.head())


def _parse_valor(v):
    if pd.isna(v) or v == '':
        return None
    s = str(v).strip().replace('R$', '').strip()
    if ',' in s:
        s = s.replace('.', '').replace(',', '.')
    return float(s)

def comparacao_valores(valor_sap, valor_sped):
    valor_sap = _parse_valor(valor_sap)
    valor_sped = _parse_valor(valor_sped)

    sap_vazio = valor_sap is None
    sped_vazio = valor_sped is None

    if sped_vazio and not sap_vazio:
        return 0, "apenas_sap"
    elif sap_vazio and not sped_vazio:
        return 0, "apenas_sped"
    elif sap_vazio and sped_vazio:
        return 0, "sem_valor"
    elif valor_sped > valor_sap:
        return valor_sped - valor_sap, "complemento"
    elif valor_sap > valor_sped:
        return valor_sap - valor_sped, "estorno"
    else:
        return 0, "ok"


correlacao_descricao_sped_sap = pd.DataFrame({
    "SAP":[
        "Credito PIS-COFINS s Depreciação",
        "Credito PIS-COFINS s Aluguéis das unidades",
        "Credito PIS-COFINS s Manutenção e conservação das unidades"
    ],
    

    "SPED":[
        "Depreciacao",
        "ALUGUÉIS DE PRÉDIOS",
        "Manutenção e conservação das unidades"

    ]

})

def converte(coluna):
    return coluna.str.replace('.', '').str.replace(',', '.').astype(float)



In [23]:
df_depreciacao_sped = df_f120[df_f120['DESC_BEM_IMOB'] == "Depreciacao"]
df_depreciacao_sped[['VL_PIS', 'VL_COFINS']] = df_depreciacao_sped[['VL_PIS', 'VL_COFINS']].apply(converte)
df_depreciacao_sped = df_depreciacao_sped.groupby('DESC_BEM_IMOB', as_index=False)[['VL_PIS', 'VL_COFINS']].sum()

df_depreciacao_sap = sap[sap['Observações'] == 'Credito PIS-COFINS s Depreciação']
df_depreciacao_sap['Valor'] = df_depreciacao_sap['Valor'].apply(_parse_valor)
df_depreciacao_sap = df_depreciacao_sap.pivot(
    index="Observações",
    columns="Imposto",
    values="Valor"
).rename(columns={
    "PIS":"VL_PIS",
    "COFINS": "VL_COFINS"
}).reset_index()

print(df_depreciacao_sped)
print(df_depreciacao_sap)

comparacao_pis = comparacao_valores(df_depreciacao_sped['VL_PIS'].iloc[0], df_depreciacao_sap['VL_PIS'].iloc[0])
comparacao_cofins = comparacao_valores(df_depreciacao_sped['VL_COFINS'].iloc[0], df_depreciacao_sap['VL_PIS'].iloc[0])

print(comparacao_pis)
print(comparacao_cofins)


  DESC_BEM_IMOB   VL_PIS  VL_COFINS
0   Depreciacao  3072.33   14151.36
Imposto                       Observações  VL_COFINS   VL_PIS
0        Credito PIS-COFINS s Depreciação   13173.98  2860.14
(212.19000000000005, 'estorno')
(11291.220000000001, 'estorno')


In [29]:
dados = extrai_dados_sped(r"C:\Users\luis.moura\Documents\GitHub\projeto-confronto-ajuste-sped\sped_test\PISCOFINS_20260301_20260331_04784935000167_Original_20260415164738_3B13D02B5763C278137AB3286018AD0488132796.txt")
print(dados)

dados["F100"].to_excel("f100.xlsx", index=False)

{'0000':     REG COD_VER TIPO_ESCRIT IND_SIT_ESP NUM_REC_ANTERIOR    DT_INI    DT_FIN  \
0  0000     006           0                               01032026  31032026   

                                 NOME            CNPJ  UF  COD_MUN SUFRAMA  \
0  CENTRAL IRRIGACAO LTDA - (Goiania)  04784935000167  GO  5208707           

  IND_NAT_PJ IND_ATIV  
0         00        0  , 'C100':       REG IND_OPER IND_EMIT COD_PART COD_MOD COD_SIT  SER NUM_DOC  \
0    C100        0        0               55      02  001   38650   
1    C100        0        0               55      02  001   38701   
2    C100        0        0               55      02  001   38743   
3    C100        1        0               55      02  001   38649   
4    C100        1        0               55      02  001   38673   
..    ...      ...      ...      ...     ...     ...  ...     ...   
796  C100        0        1   F04912      55      00  001      20   
797  C100        0        1   F04953      55      00  001     74